In [20]:
import os
from lxml import etree
from pprint import pprint
from mergedeep import merge, Strategy

def update_xml(dst, src):
    if src is None:
        return
    
    for key in dst.attrib.keys():
        if key in src.attrib and dst.attrib[key] != src.attrib[key]:
            dst.attrib[key] = src.attrib[key]

    for dst_child in dst:
        name = dst_child.get('name')
        update_xml(dst_child, src.find(f".//{dst_child.tag}[@name='{name}']"))

In [ ]:
model_xml_env = etree.parse(r'model_xml_env.xml')
model_xml_env_root = model_xml_env.getroot()

model_xml_demo_0 = etree.parse(r'model_xml_demo_279.xml')
model_xml_demo_0_root = model_xml_demo_0.getroot()

In [22]:
update_xml(model_xml_env_root, model_xml_demo_0_root)
pprint(etree.tostring(model_xml_env_root, pretty_print=True).decode())
etree.ElementTree(model_xml_env_root).write(r'model_xml_env_updated.xml', pretty_print=True)

('<mujoco model="base">\n'
 '  <compiler angle="radian" meshdir="meshes/" autolimits="true"/>\n'
 '  <option impratio="20" density="1.2" viscosity="2e-05" cone="elliptic"/>\n'
 '  <size njmax="5000" nconmax="5000"/>\n'
 '  <visual>\n'
 '    <map znear="0.001"/>\n'
 '  </visual>\n'
 '  <asset>\n'
 '    <texture type="skybox" builtin="gradient" rgb1="0.9 0.9 1" rgb2="0.2 0.3 '
 '0.4" width="256" height="1536"/>\n'
 '    <texture type="2d" name="texplane" '
 'file="/data1/hanfang_data/Documents/robosuite/robosuite/models/assets/arenas/../textures/light-gray-floor-tile.png"/>\n'
 '    <texture type="cube" name="tex-steel-brushed" '
 'file="/data1/hanfang_data/Documents/robosuite/robosuite/models/assets/arenas/../textures/steel-brushed.png"/>\n'
 '    <texture type="2d" name="tex-light-gray-plaster" '
 'file="/data1/hanfang_data/Documents/robosuite/robosuite/models/assets/arenas/../textures/light-gray-plaster.png"/>\n'
 '    <texture type="2d" name="tex-light-wood" '
 'file="/data1/hanfang_

In [3]:
def xml2dict(node_xml):
    '''
    convert an XML node (and all of its children) into a nested
    dictionary
    '''
    node_dict = {}
    if node_xml.attrib:
        node_dict['attributes'] = node_xml.attrib

        if "/home/robot/installed_libraries" in node_xml.attrib.get('file', ''):
            node_dict['attributes']['file'] = node_dict['attributes']['file'].replace("/home/robot/installed_libraries", os.getcwd()[:-20])

        if node_xml.attrib.get('name', '') == 'Cereal_tex-cereal':
            node_dict['attributes']['type'] = '2d'
    
    node_dict['elements'] = [{child.tag: xml2dict(child)} for child in node_xml]
    if not node_dict['elements']:
        node_dict.pop('elements')

    return node_dict

In [4]:
def dict2xml(root_tag, node_dict):
    node_xml = ET.Element(root_tag, node_dict.get('attributes', {}))
    for element in node_dict.get('elements', []):
        for tag, child_dict in element.items():
            node_xml.append(dict2xml(tag, child_dict))

    return node_xml

In [5]:
model_xml_env_dict = xml2dict(model_xml_env_root)
model_xml_demo_0_dict = xml2dict(model_xml_demo_0_root)

merge(model_xml_env_dict, model_xml_demo_0_dict, strategy=Strategy.REPLACE) 
pprint(model_xml_env_dict)

model_xml_env_updated = dict2xml('mujoco', model_xml_env_dict)
ET.ElementTree(model_xml_env_updated).write('model_xml_env_updated.xml', encoding='unicode', xml_declaration=True)

{'attributes': {'model': 'base'},
 'elements': [{'compiler': {'attributes': {'angle': 'radian',
                                           'meshdir': 'meshes/'}}},
              {'option': {'attributes': {'cone': 'elliptic',
                                         'density': '1.2',
                                         'impratio': '20',
                                         'viscosity': '2e-05'}}},
              {'size': {'attributes': {'nconmax': '5000', 'njmax': '5000'}}},
              {'visual': {'elements': [{'map': {'attributes': {'znear': '0.001'}}}]}},
              {'asset': {'elements': [{'texture': {'attributes': {'builtin': 'gradient',
                                                                  'height': '1536',
                                                                  'rgb1': '0.9 '
                                                                          '0.9 '
                                                                          '1',
            